In [1]:
import pandas as pd

import meteostat as ms
from datetime import datetime
from pathlib import Path

In [2]:
start = datetime(2025, 1, 1)
end = datetime(2025, 1, 3, 23, 59)

la = ms.Point(34.0522, -118.2437)

# Find nearby stations
stations = ms.stations.nearby(la, limit=4)

# Get hourly data from those stations
ts = ms.hourly(stations, start, end)

# Interpolate to the point
weather = ms.interpolate(ts, la).fetch()

print(type(weather))
print(weather.head())
print(weather.columns.tolist())

<class 'pandas.DataFrame'>
                     temp  rhum  prcp  wdir  wspd    pres  cldc  coco
time                                                                 
2025-01-01 00:00:00  16.3    60   0.0   253   7.7  1016.4     8     3
2025-01-01 01:00:00  14.3    68   0.0   260   7.7  1016.9     7     3
2025-01-01 02:00:00  12.9    78   0.0   275   6.3  1017.4     4     3
2025-01-01 03:00:00  12.0    82   0.0   276   4.0  1017.6     0     3
2025-01-01 04:00:00  11.3    87   0.0   303   3.7  1017.9     1     1
['temp', 'rhum', 'prcp', 'wdir', 'wspd', 'pres', 'cldc', 'coco']


In [3]:
start = datetime(2025, 1, 1)
end = datetime(2025, 12, 31, 23, 59)

points = {
    "la": ms.Point(34.0522, -118.2437),
    "sf": ms.Point(37.7749, -122.4194),
    "sd": ms.Point(32.7157, -117.1611),
}

weather_dfs = {}

for name, point in points.items():
    stations = ms.stations.nearby(point, limit=4)
    ts = ms.hourly(stations, start, end)
    df = ms.interpolate(ts, point).fetch().reset_index()
    df = df.rename(columns={"time": "datetime"})
    weather_dfs[name] = df
    print(name, df.shape)
    print(df.head(2))

la (8760, 10)
             datetime  temp  rhum  prcp  wdir  wspd  wpgt    pres  cldc  coco
0 2025-01-01 00:00:00  16.3    60   0.0   253   7.7  <NA>  1016.4     8     3
1 2025-01-01 01:00:00  14.3    68   0.0   260   7.7  <NA>  1016.9     7     3
sf (8760, 10)
             datetime  temp  rhum  prcp  wdir  wspd  wpgt    pres  cldc  coco
0 2025-01-01 00:00:00  12.1    60   0.0   310   5.7  <NA>  1021.8     7     3
1 2025-01-01 01:00:00  11.8    60   0.0   310   8.2  <NA>  1021.7     6     3
sd (8760, 10)
             datetime  temp  rhum  prcp  wdir  wspd  wpgt    pres  cldc  coco
0 2025-01-01 00:00:00  15.0    72   0.0   320   7.6  <NA>  1017.4     6     3
1 2025-01-01 01:00:00  13.9    74   0.0   320   7.6  <NA>  1017.5     7     3


## Weather Feature Selection

- The Meteostat dataset includes many variables (temp, humidity, precipitation, wind, pressure, cloud cover, etc.).
- For this initial model, we focus on a small set of high-signal features:
  - **temp** → primary driver of electricity demand  
  - **rhum** → affects perceived temperature  
  - **prcp** → captures storm/cloud effects  
  - **wspd** → reflects broader weather patterns  

- This simplification helps to:
  - Focus on the most meaningful drivers of demand  
  - Avoid multicollinearity between overlapping features  
  - Reduce noise and risk of overfitting  
  - Keep the model interpretable and easy to debug  

In [5]:
def prep_weather(df, prefix):
    keep = ["datetime", "temp", "rhum", "prcp", "wspd"]
    keep = [c for c in keep if c in df.columns]
    df = df[keep].copy()
    df = df.rename(columns={c: f"{prefix}_{c}" for c in df.columns if c != "datetime"})
    return df

weather_la = prep_weather(weather_dfs["la"], "la")
weather_sf = prep_weather(weather_dfs["sf"], "sf")
weather_sd = prep_weather(weather_dfs["sd"], "sd")

In [6]:
weather_df = weather_la.merge(weather_sf, on="datetime", how="outer")
weather_df = weather_df.merge(weather_sd, on="datetime", how="outer")
weather_df = weather_df.sort_values("datetime").reset_index(drop=True)

print(weather_df.head())
print(weather_df.shape)

             datetime  la_temp  la_rhum  la_prcp  la_wspd  sf_temp  sf_rhum  \
0 2025-01-01 00:00:00     16.3       60      0.0      7.7     12.1       60   
1 2025-01-01 01:00:00     14.3       68      0.0      7.7     11.8       60   
2 2025-01-01 02:00:00     12.9       78      0.0      6.3     11.6       60   
3 2025-01-01 03:00:00     12.0       82      0.0      4.0     11.3       63   
4 2025-01-01 04:00:00     11.3       87      0.0      3.7     11.3       68   

   sf_prcp  sf_wspd  sd_temp  sd_rhum  sd_prcp  sd_wspd  
0      0.0      5.7     15.0       72      0.0      7.6  
1      0.0      8.2     13.9       74      0.0      7.6  
2      0.0     13.3     12.8       80      0.0      5.4  
3      0.0     13.4     12.8       83      0.0      0.0  
4      0.0      9.7     12.2       83      0.0      5.4  
(8760, 13)


In [7]:
folder_path = Path("../data/ems_hourly")  

# get all Excel files
files = sorted(folder_path.glob("*.xlsx"))

print(f"Found {len(files)} files")

dfs = []

for file in files:
    print(f"Loading {file.name}")
    df = pd.read_excel(file)
    dfs.append(df)

# combine into one dataframe
load_df = pd.concat(dfs, ignore_index=True)

print(load_df.shape)
load_df.head()

Found 12 files
Loading historical-ems-hourly-load-for-april-2025.xlsx
Loading historical-ems-hourly-load-for-august-2025.xlsx
Loading historical-ems-hourly-load-for-december-2025.xlsx
Loading historical-ems-hourly-load-for-february-2025.xlsx
Loading historical-ems-hourly-load-for-january-2025.xlsx
Loading historical-ems-hourly-load-for-july-2025.xlsx
Loading historical-ems-hourly-load-for-june-2025.xlsx
Loading historical-ems-hourly-load-for-march-2025.xlsx
Loading historical-ems-hourly-load-for-may-2025.xlsx
Loading historical-ems-hourly-load-for-november-2025.xlsx
Loading historical-ems-hourly-load-for-october-2025.xlsx
Loading historical-ems-hourly-load-for-september-2025.xlsx
(8782, 10)


,Date,HR,PGE,SCE,SDGE,VEA,CAISO,Unnamed: 7,Spring DST change occurs on March 9 in HE2,Fall DST change occurs on 11/2/2025 at 1am
0,2025-04-01 00:00:00,1.0,9917.144076,9875.463107,2075.362681,51.403279,21919.373688,NaN,NaN,NaN
1,2025-04-01 00:00:00,2.0,9704.983094,9295.090042,2043.725304,50.381094,21094.179853,NaN,NaN,NaN
2,2025-04-01 00:00:00,3.0,9596.863617,9057.183108,1937.794120,50.715697,20642.556684,NaN,NaN,NaN
3,2025-04-01 00:00:00,4.0,9479.991969,8903.228792,1880.379813,52.398983,20315.999813,NaN,NaN,NaN
4,2025-04-01 00:00:00,5.0,9581.070999,9017.621300,1877.684550,56.152298,20532.529551,NaN,NaN,NaN


In [8]:
print(load_df.columns.tolist())

['Date', 'HR', 'PGE', 'SCE', 'SDGE', 'VEA', 'CAISO', 'Unnamed: 7', 'Spring DST change occurs on March 9 in HE2', 'Fall DST change occurs on 11/2/2025 at 1am']


In [9]:
load_df = load_df.drop(columns=[
    "Unnamed: 7",
    "Spring DST change occurs on March 9 in HE2",
    "Fall DST change occurs on 11/2/2025 at 1am"
], errors="ignore")

In [10]:
load_df["Date"].astype(str).value_counts().head(20)

Date
2025-11-02 00:00:00    25
2025-04-01 00:00:00    24
2025-04-02 00:00:00    24
2025-04-03 00:00:00    24
2025-04-04 00:00:00    24
2025-04-05 00:00:00    24
2025-04-06 00:00:00    24
2025-04-07 00:00:00    24
2025-04-08 00:00:00    24
2025-04-09 00:00:00    24
2025-04-10 00:00:00    24
2025-04-11 00:00:00    24
2025-04-12 00:00:00    24
2025-04-13 00:00:00    24
2025-04-14 00:00:00    24
2025-04-15 00:00:00    24
2025-04-16 00:00:00    24
2025-04-17 00:00:00    24
2025-04-18 00:00:00    24
2025-04-19 00:00:00    24
Name: count, dtype: int64

In [11]:
load_df[~pd.to_datetime(load_df["Date"], errors="coerce").notna()][["Date", "HR"]].head(20)

,Date,HR
720,NaN,NaN
721,CAISO Public,NaN
1466,NaN,NaN
1467,CAISO Public,NaN
2212,NaN,NaN
2213,CAISO Public,NaN
2886,NaN,NaN
2887,CAISO Public,NaN
4376,NaN,NaN
4377,CAISO Public,NaN


In [12]:
load_df["Date"] = pd.to_datetime(load_df["Date"], errors="coerce")
load_df["HR"] = pd.to_numeric(load_df["HR"], errors="coerce")

load_df = load_df.dropna(subset=["Date", "HR"]).copy()

load_df["datetime"] = load_df["Date"] + pd.to_timedelta(load_df["HR"] - 1, unit="h")

load_df = load_df.rename(columns={"CAISO": "load_mw"})
load_df = load_df[["datetime", "load_mw"]]
load_df = load_df.sort_values("datetime").reset_index(drop=True)

print(load_df.head())
print(load_df.shape)
print(load_df["datetime"].duplicated().sum())

             datetime       load_mw
0 2025-01-01 00:00:00  21768.585137
1 2025-01-01 01:00:00  21111.893379
2 2025-01-01 02:00:00  20358.821469
3 2025-01-01 03:00:00  19926.846964
4 2025-01-01 04:00:00  19916.727203
(8760, 2)
1


In [13]:
model_df = load_df.merge(weather_df, on="datetime", how="left")

In [14]:
print("Shape:", model_df.shape)

print("\nMissing values:")
print(model_df.isna().sum().sort_values(ascending=False).head(10))

print("\nTime gaps:")
print(model_df["datetime"].diff().value_counts().head())

print("\nPreview:")
print(model_df.head())

Shape: (8760, 14)

Missing values:
sf_prcp     6
sd_prcp     2
datetime    0
load_mw     0
la_temp     0
la_rhum     0
la_prcp     0
la_wspd     0
sf_temp     0
sf_rhum     0
dtype: int64

Time gaps:
datetime
0 days 01:00:00    8757
0 days 02:00:00       1
0 days 00:00:00       1
Name: count, dtype: int64

Preview:
             datetime       load_mw  la_temp  la_rhum  la_prcp  la_wspd  \
0 2025-01-01 00:00:00  21768.585137     16.3       60      0.0      7.7   
1 2025-01-01 01:00:00  21111.893379     14.3       68      0.0      7.7   
2 2025-01-01 02:00:00  20358.821469     12.9       78      0.0      6.3   
3 2025-01-01 03:00:00  19926.846964     12.0       82      0.0      4.0   
4 2025-01-01 04:00:00  19916.727203     11.3       87      0.0      3.7   

   sf_temp  sf_rhum  sf_prcp  sf_wspd  sd_temp  sd_rhum  sd_prcp  sd_wspd  
0     12.1       60      0.0      5.7     15.0       72      0.0      7.6  
1     11.8       60      0.0      8.2     13.9       74      0.0      7.6  
2   

In [15]:
model_df.to_csv("../data/model_input_hourly.csv", index=False)

In [16]:
# create daily dataframe

daily_source_df = model_df.copy()
daily_source_df["datetime"] = pd.to_datetime(daily_source_df["datetime"])
daily_source_df = daily_source_df.sort_values("datetime").reset_index(drop=True)

# fill small missing precipitation values with 0
for col in ["la_prcp", "sf_prcp", "sd_prcp"]:
    if col in daily_source_df.columns:
        daily_source_df[col] = daily_source_df[col].fillna(0)

# create daily features
daily_df = (
    daily_source_df
    .resample("D", on="datetime")
    .agg({
        "load_mw": ["mean", "max", "min", "std"],

        "la_temp": ["mean", "max", "min", "std"],
        "la_rhum": ["mean", "max", "min", "std"],
        "la_prcp": ["sum", "max"],
        "la_wspd": ["mean", "max", "std"],

        "sf_temp": ["mean", "max", "min", "std"],
        "sf_rhum": ["mean", "max", "min", "std"],
        "sf_prcp": ["sum", "max"],
        "sf_wspd": ["mean", "max", "std"],

        "sd_temp": ["mean", "max", "min", "std"],
        "sd_rhum": ["mean", "max", "min", "std"],
        "sd_prcp": ["sum", "max"],
        "sd_wspd": ["mean", "max", "std"],
    })
)

# flatten multi-index column names
daily_df.columns = [f"{col[0]}_{col[1]}" for col in daily_df.columns]

# bring datetime back as a column
daily_df = daily_df.reset_index()

# create same-day target
daily_df["target_load_mw_mean"] = daily_df["load_mw_mean"]

# create calendar columns for EDA / modeling later
daily_df["year"] = daily_df["datetime"].dt.year
daily_df["month"] = daily_df["datetime"].dt.month
daily_df["day"] = daily_df["datetime"].dt.day
daily_df["day_of_week"] = daily_df["datetime"].dt.dayofweek
daily_df["day_name"] = daily_df["datetime"].dt.day_name()
daily_df["day_of_year"] = daily_df["datetime"].dt.dayofyear
daily_df["is_weekend"] = daily_df["day_of_week"].isin([5, 6]).astype(int)

# inspect result
print("Daily shape:", daily_df.shape)
print("\nMissing values:")
print(daily_df.isna().sum()[daily_df.isna().sum() > 0])

print("\nPreview:")
print(daily_df.head())

Daily shape: (365, 52)

Missing values:
Series([], dtype: int64)

Preview:
    datetime  load_mw_mean   load_mw_max   load_mw_min  load_mw_std  \
0 2025-01-01  22574.940652  24870.967744  19916.727203  1631.902702   
1 2025-01-02  24140.178717  27473.091131  19700.929643  2522.330349   
2 2025-01-03  24641.971448  27373.142624  21211.978156  1933.071405   
3 2025-01-04  23242.359045  26109.311246  20251.253931  1748.794413   
4 2025-01-05  22584.144627  25099.710522  20115.855130  1519.459334   

   la_temp_mean  la_temp_max  la_temp_min  la_temp_std  la_rhum_mean  ...  \
0        11.875         19.2          6.8     4.169272        68.125  ...   
1     13.683333         23.6          7.7      5.34071     56.958333  ...   
2       13.5875         21.4          9.4     3.157092     63.458333  ...   
3     12.391667         17.3          8.3     2.891504        80.875  ...   
4     13.295833         23.3          7.3     5.407039        50.875  ...   

   sd_wspd_max  sd_wspd_std  target

In [17]:
daily_df.to_csv("../data/model_input_daily.csv", index=False)

## Data Wrangling Summary

- Collected hourly weather data (LA, SF, SD) and kept key features: temp, humidity, precipitation, wind.
- Cleaned and combined 12 months of CAISO load data; built a proper hourly `datetime` column.
- Merged weather + load into a single hourly dataset (~8,760 rows) with minimal missing data.

- Aggregated to daily level using summary stats (mean, max, min, std).
- Created target variable: daily average load (`target_load_mw_mean`).
- Added calendar features (month, day_of_week, day_of_year, weekend flag).

- Final outputs:
  - `model_input_hourly.csv` (raw time series)
  - `model_input_daily.csv` (model-ready dataset)

**Takeaway:**  
Clean, aligned dataset combining statewide energy demand with multi-city weather + time features, ready for modeling daily CA load.